# Continuation Ledger — build 1

**Dean Kulik — QuHarmonics Research Group** · CC BY-NC 4.0

**BUILD** the module `continuation_ledger`. **TEST** it against four contracts declared before running.

Input: a system (states, actions, transition map) and an insufficient reader $R$.
Output: the minimum state the reader must add so that every continuation is determined:

$$R^\star(x_1)=R^\star(x_2)\;\Longrightarrow\;R^\star(T_a x_1)=R^\star(T_a x_2)\quad\text{for every action }a.$$

`close_reader` computes the coarsest such $R^\star$ that refines $R$ (Moore partition refinement). No refinement coarser than $R^\star$ satisfies the condition, so the ledger it adds is the minimum.

In [1]:
# ============================================================ continuation_ledger (the module)
import itertools, math, random
from collections import defaultdict, deque

class Ledger:
    """Result of closing a reader. All fields are computed, none are supplied."""
    def __repr__(self):
        return (f"Ledger(states={len(self.states)}, reader_fibers={len(self.reader_fibers)}, "
                f"refined_classes={self.n_classes}, required_capacity={self.required_capacity}, "
                f"ledger_bits={self.ledger_bits}, violations={self.violations})")

def close_reader(states, actions, T, R):
    """Coarsest refinement R* of reader R that is closed under every action.
    T(x, a) returns the successor state, or None if a is not legal at x."""
    L = Ledger(); L.states = list(states); L.actions = list(actions)
    idx = {x: i for i, x in enumerate(L.states)}
    succ = []
    for x in L.states:
        row = []
        for a in L.actions:
            y = T(x, a); row.append(idx[y] if y is not None else -1)
        succ.append(row)
    rv = [R(x) for x in L.states]
    first = {}
    blk = [first.setdefault(v, len(first)) for v in rv]
    while True:                                   # refine until stable
        sig = {}
        new = [sig.setdefault((blk[i], tuple(blk[j] if j >= 0 else -1 for j in succ[i])), len(sig))
               for i in range(len(L.states))]
        if len(sig) == len(set(blk)):
            blk = new; break
        blk = new
    L.block, L.reader_value, L._succ, L._idx = blk, rv, succ, idx
    L.n_classes = len(set(blk))
    # reader fibers and the classes hidden inside each
    fib = defaultdict(list)
    for i, v in enumerate(rv): fib[v].append(i)
    L.reader_fibers = dict(fib)
    L.future_classes = {v: sorted({blk[i] for i in ids}) for v, ids in fib.items()}
    L.required_capacity = max(len(c) for c in L.future_classes.values())
    L.ledger_bits = math.ceil(math.log2(L.required_capacity)) if L.required_capacity > 1 else 0
    # minimum labels: the index of a state's class inside its own fiber
    L.minimum_labels = {L.states[i]: L.future_classes[rv[i]].index(blk[i]) for i in range(len(L.states))}
    L.refined_reader = lambda x: (R(x), L.minimum_labels[x])
    # reduced transitions on classes, and the closure check
    red = defaultdict(set)
    for i in range(len(L.states)):
        for k, j in enumerate(succ[i]):
            red[(blk[i], k)].add(blk[j] if j >= 0 else None)
    L.violations = sum(1 for v in red.values() if len(v) > 1)
    L.reduced_transitions = {key: next(iter(v)) for key, v in red.items()}
    L.class_value = {blk[i]: rv[i] for i in range(len(L.states))}
    # one shortest witness per split fiber: same reader value, different lawful future
    L.branching_counterexamples = []
    for v, ids in fib.items():
        cl = L.future_classes[v]
        if len(cl) < 2: continue
        i1 = next(i for i in ids if blk[i] == cl[0]); i2 = next(i for i in ids if blk[i] == cl[1])
        L.branching_counterexamples.append(_witness(L, i1, i2))
    return L

def _witness(L, i1, i2):
    """shortest action word after which the ORIGINAL reader tells i1 and i2 apart"""
    seen = {(i1, i2)}; q = deque([(i1, i2, ())])
    while q:
        a, b, w = q.popleft()
        if a < 0 or b < 0 or L.reader_value[a] != L.reader_value[b]:
            return (L.states[i1], L.states[i2], w)
        for k in range(len(L.actions)):
            p = (L._succ[a][k], L._succ[b][k])
            if p not in seen:
                seen.add(p); q.append((p[0], p[1], w + (L.actions[k],)))
    return (L.states[i1], L.states[i2], None)

def peek(L, cls):
    """what the original reader sees in a refined class"""
    return L.class_value[cls]

def poke(L, cls, word):
    """apply an action word to a refined class through the reduced table; no history replay"""
    k = {a: n for n, a in enumerate(L.actions)}
    for a in word:
        cls = L.reduced_transitions[(cls, k[a])]
        if cls is None: return None
    return cls

def label(L, x):
    return L.block[L._idx[x]]

def future_classes(readings, completions, out):
    """Myhill-Nerode classes of readings: equal iff every completion gives the same output on the
    positions still pending. Positions already determined count as emitted, not carried."""
    classes = set()
    for r in readings:
        rows = [out(c) for c in completions(r)]
        pend = tuple(i for i in range(len(rows[0])) if len({o[i] for o in rows}) > 1)
        classes.add((pend, tuple(tuple(o[i] for i in pend) for o in rows)))
    return classes

def transport(sources, image, targets):
    """write a source into a receiver: image(x) -> target or None (does not fit).
    Returns excluded sources, invented targets (never written) and identified targets."""
    pre = defaultdict(list); excluded = 0
    for x in sources:
        t = image(x)
        if t is None: excluded += 1
        else: pre[t].append(x)
    invented = sum(1 for t in targets if t not in pre)
    identified = sum(1 for t in targets if len(pre.get(t, ())) > 1)
    verdict = "exclusion" if excluded else ("invention" if invented else "faithful")
    return dict(excluded=excluded, invented=invented, identified=identified, verdict=verdict)

RESULTS = []
def contract(name, ok, detail):
    RESULTS.append((name, ok, detail)); print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")
print("continuation_ledger loaded")

continuation_ledger loaded


---
## Test 1 — a deliberately lossy machine

A 2-bit counter plus one junk bit that never affects anything. Actions: `inc` (counter + 1 mod 4) and `flip` (toggle the junk bit). The reader sees only the counter's high bit.

**Contract:** the runtime splits each fiber on the counter's low bit, which is a 1-bit ledger. It must **not** add the junk bit, because that distinction has no effect on any future. `poke` through the reduced table must agree with running the machine.

In [2]:
states = [(c, j) for c in range(4) for j in (0, 1)]
T1 = lambda x, a: ((x[0] + 1) % 4, x[1]) if a == "inc" else (x[0], x[1] ^ 1)
L1 = close_reader(states, ["inc", "flip"], T1, R=lambda x: x[0] // 2)
print(L1)
print("witnesses (same reading, different future):", L1.branching_counterexamples)
low_bit = all(L1.minimum_labels[x] == L1.minimum_labels[(x[0] % 2, 0) if x[0] < 2 else (2 + x[0] % 2, 0)] and
              len({L1.minimum_labels[y] for y in states if y[0] // 2 == x[0] // 2 and y[0] % 2 == x[0] % 2}) == 1
              for x in states) and len({(x[0] // 2, L1.minimum_labels[x]) for x in states}) == 4
junk_merged = all(label(L1, (c, 0)) == label(L1, (c, 1)) for c in range(4))
rng = random.Random(1); ok = 0
for _ in range(2000):
    x = rng.choice(states); w = tuple(rng.choice(["inc", "flip"]) for _ in range(rng.randint(0, 12))); y = x
    for a in w: y = T1(y, a)
    ok += poke(L1, label(L1, x), w) == label(L1, y)
contract("T1 ledger = 1 bit", L1.ledger_bits == 1 and L1.violations == 0, f"ledger_bits={L1.ledger_bits}, violations={L1.violations}")
contract("T1 ledger is the low counter bit", low_bit, f"classes = (high bit, low bit): {low_bit}")
contract("T1 junk bit not added", junk_merged, f"states differing only in the junk bit share a class: {junk_merged}")
contract("T1 poke = run", ok == 2000, f"{ok}/2000 random words")

Ledger(states=8, reader_fibers=2, refined_classes=4, required_capacity=2, ledger_bits=1, violations=0)
witnesses (same reading, different future): [((0, 0), (1, 0), ('inc',)), ((2, 0), (3, 0), ('inc',))]
PASS  T1 ledger = 1 bit: ledger_bits=1, violations=0
PASS  T1 ledger is the low counter bit: classes = (high bit, low bit): True
PASS  T1 junk bit not added: states differing only in the junk bit share a class: True
PASS  T1 poke = run: 2000/2000 random words


---
## Test 2 — binary adder

The states are every input history of bit pairs $(a_i, b_i)$ read least-significant-first, up to length 6. There are 5461 of them, and the machine is never told the word "carry". The reader sees only the sum bits written so far.

**Contract:** the runtime finds a 1-bit ledger. On every history that still has a future, its classes coincide exactly with (written bits, carry). On histories with no future (length 6) it adds nothing.

In [3]:
LEN = 6
def written(h):
    c, out = 0, []
    for a, b in h:
        v = a + b + c; out.append(v % 2); c = v // 2
    return tuple(out), c
hist = [h for k in range(LEN + 1) for h in itertools.product([(0, 0), (0, 1), (1, 0), (1, 1)], repeat=k)]
T2 = lambda h, a: h + (a,) if len(h) < LEN else None
L2 = close_reader(hist, [(0, 0), (0, 1), (1, 0), (1, 1)], T2, R=lambda h: written(h)[0])
print(L2)
print("a witness:", L2.branching_counterexamples[-1])
live = [h for h in hist if len(h) < LEN]; dead = [h for h in hist if len(h) == LEN]
same = (len({label(L2, h) for h in live}) == len({(written(h)[0], written(h)[1]) for h in live}) ==
        len({(label(L2, h), written(h)[1]) for h in live}))
nothing_at_end = all(len(L2.future_classes[written(h)[0]]) == 1 for h in dead)
rng = random.Random(2); ok = 0
for _ in range(2000):
    h = rng.choice(live); w = tuple(rng.choice([(0, 0), (0, 1), (1, 0), (1, 1)]) for _ in range(rng.randint(0, LEN - len(h))))
    ok += poke(L2, label(L2, h), w) == label(L2, h + w) and peek(L2, label(L2, h + w)) == written(h + w)[0]
contract("T2 ledger = 1 bit", L2.ledger_bits == 1 and L2.violations == 0, f"ledger_bits={L2.ledger_bits}, violations={L2.violations}")
contract("T2 ledger is the carry", same, f"classes on live histories == (written bits, carry): {same}")
contract("T2 no ledger without a future", nothing_at_end, f"length-{LEN} fibers unsplit: {nothing_at_end}")
contract("T2 peek/poke = run", ok == 2000, f"{ok}/2000 random continuations")

Ledger(states=5461, reader_fibers=127, refined_classes=184, required_capacity=2, ledger_bits=1, violations=0)
a witness: (((0, 1), (0, 1), (0, 1), (0, 1), (0, 0)), ((0, 1), (0, 1), (0, 1), (0, 1), (1, 1)), ((0, 0),))
PASS  T2 ledger = 1 bit: ledger_bits=1, violations=0
PASS  T2 ledger is the carry: classes on live histories == (written bits, carry): True
PASS  T2 no ledger without a future: length-6 fibers unsplit: True
PASS  T2 peek/poke = run: 2000/2000 random continuations


---
## Test 3 — reader-cut adder: when do gap ledgers factor?

This is the corpus machine from "Carry Capacity and the Conditional Cut Law". It has radix $b$ and $m$ operands, and $t_q$ is the digit sum at position $q$. Each step computes $s_q = (t_q + c_q) \bmod b$ and $c_{q+1} = \lfloor (t_q + c_q)/b \rfloor$. A reader reads the positions in a cut $S$, and its state count is $F(S)$, computed by `future_classes`. The ledgers are *gap-local* when $F(S)$ equals the product over the maximal read runs.

**Contract:** for each $(b, m)$, the smallest gap at which factoring holds for every cut equals $\lceil \log_b m \rceil$. The failure counts must reproduce the corpus exactly: 18 for $(2,3)$, 27 for $(2,4)$, 2 for $(3,4)$ and 2 for $(4,5)$.

In [4]:
def line_out(t, b):
    c, s = 0, []
    for v in t:
        v += c; s.append(v % b); c = v // b
    return tuple(s)
def runs_of(S, n):
    out, cur = [], None
    for p in range(n):
        if p in S:
            cur = [p, p] if cur is None else [cur[0], p]
        elif cur is not None:
            out.append(tuple(cur)); cur = None
    if cur is not None: out.append(tuple(cur))
    return out
def gaps_of(S, n):
    U = [p for p in range(n) if p not in S]
    return [len(list(g)) for _, g in itertools.groupby(enumerate(U), lambda t: t[1] - t[0])]
def make_F(n, b, m):
    hi = m * (b - 1) + 1; memo = {}
    def F(S):
        S = tuple(sorted(S))
        if S not in memo:
            U = [p for p in range(n) if p not in S]
            def completions(r):
                for uv in itertools.product(range(hi), repeat=len(U)):
                    t = [0] * n
                    for p, v in zip(S, r): t[p] = v
                    for p, v in zip(U, uv): t[p] = v
                    yield t
            memo[S] = len(future_classes(itertools.product(range(hi), repeat=len(S)), completions, lambda t: line_out(t, b)))
        return memo[S]
    return F
CORPUS_FAILS = {(2, 2): 0, (2, 3): 18, (2, 4): 27, (3, 3): 0, (3, 4): 2, (4, 5): 2, (5, 5): 0}
print(f"{'b':>2} {'m':>2} {'n':>2} {'ceil(log_b m)':>13} {'observed':>9} {'fails':>6} {'corpus':>7}")
all_ok = True
for (b, m, n) in [(2, 2, 6), (2, 3, 6), (2, 4, 6), (3, 3, 5), (3, 4, 4), (4, 5, 4), (5, 5, 3)]:
    F = make_F(n, b, m); by = defaultdict(lambda: [0, 0])
    for k in range(n + 1):
        for S in itertools.combinations(range(n), k):
            prod = 1
            for (l, r) in runs_of(set(S), n): prod *= F(range(l, r + 1))
            gs = gaps_of(S, n); g = min(gs) if gs else n
            by[g][0 if prod == F(S) else 1] += 1
    obs = min(g for g in sorted(by) if all(by[gg][1] == 0 for gg in by if gg >= g))
    pred = math.ceil(math.log(m, b) - 1e-12); fails = sum(v[1] for v in by.values())
    ok = obs == pred and fails == CORPUS_FAILS[(b, m)]; all_ok &= ok
    print(f"{b:>2} {m:>2} {n:>2} {pred:>13} {obs:>9} {fails:>6} {CORPUS_FAILS[(b, m)]:>7}  {'ok' if ok else 'MISS'}")
contract("T3 factoring threshold", all_ok, "observed gap threshold == ceil(log_b m) and failure counts == corpus, 7 radix-operand pairs")

 b  m  n ceil(log_b m)  observed  fails  corpus
 2  2  6             1         1      0       0  ok
 2  3  6             2         2     18      18  ok
 2  4  6             2         2     27      27  ok
 3  3  5             1         1      0       0  ok
 3  4  4             2         2      2       2  ok
 4  5  4             2         2      2       2  ok
 5  5  3             1         1      0       0  ok
PASS  T3 factoring threshold: observed gap threshold == ceil(log_b m) and failure counts == corpus, 7 radix-operand pairs


---
## Test 4 — arena contraction: receiver capacity against the written ledger

The arena is $K = 3$ tokens on a 6-cycle, with exclusion (at most one token per vertex). Contract the edge $(0,1)$ into one site $w$ of capacity $c$. The runtime computes $\ell$, the most the source ever writes into $w$; it is not told $\ell$. It then writes every source configuration into the receiver.

**Contract:** $c<\ell$ gives exclusion (sources with no image). $c=\ell$ gives faithful transport: nothing excluded, nothing invented, and merged states counted as identification. $c>\ell$ gives invention (receiver states no source writes).

In [5]:
n, K, u, v = 6, 3, 0, 1
src = list(itertools.combinations(range(n), K))
rest_sites = [x for x in range(n) if x not in (u, v)]
load = lambda S: sum(1 for x in S if x in (u, v))
ell = max(load(S) for S in src)
print(f"runtime-computed ledger into the merged site: ell = {ell}")
verdicts = {}
for c in (1, 2, 3):
    targets = [(r, k) for k in range(c + 1) for r in itertools.combinations(rest_sites, K - k)]
    img = lambda S, c=c: (tuple(x for x in S if x not in (u, v)), load(S)) if load(S) <= c else None
    res = transport(src, img, targets); verdicts[c] = res["verdict"]
    print(f"  capacity c={c}: {res}")
expect = {c: ("exclusion" if c < ell else "faithful" if c == ell else "invention") for c in (1, 2, 3)}
contract("T4 capacity rule", verdicts == expect, f"verdicts {verdicts} == expected {expect}")

runtime-computed ledger into the merged site: ell = 2
  capacity c=1: {'excluded': 4, 'invented': 0, 'identified': 6, 'verdict': 'exclusion'}
  capacity c=2: {'excluded': 0, 'invented': 0, 'identified': 6, 'verdict': 'faithful'}
  capacity c=3: {'excluded': 0, 'invented': 1, 'identified': 6, 'verdict': 'invention'}
PASS  T4 capacity rule: verdicts {1: 'exclusion', 2: 'faithful', 3: 'invention'} == expected {1: 'exclusion', 2: 'faithful', 3: 'invention'}


---
## Result

In [6]:
print(f"{'contract':<40} result")
print("-" * 52)
for name, ok, _ in RESULTS:
    print(f"{name:<40} {'PASS' if ok else 'FAIL'}")
print(f"\n{sum(ok for _, ok, _ in RESULTS)}/{len(RESULTS)} contracts pass")

contract                                 result
----------------------------------------------------
T1 ledger = 1 bit                        PASS
T1 ledger is the low counter bit         PASS
T1 junk bit not added                    PASS
T1 poke = run                            PASS
T2 ledger = 1 bit                        PASS
T2 ledger is the carry                   PASS
T2 no ledger without a future            PASS
T2 peek/poke = run                       PASS
T3 factoring threshold                   PASS
T4 capacity rule                         PASS

10/10 contracts pass


**Established here, nothing beyond it:**

- `close_reader` takes an insufficient reader and returns the minimum ledger: the coarsest closed refinement. With it come the split fibers, a shortest witness for each split, the reduced transitions, the required capacity and `peek`/`poke`.
- On the lossy machine it adds the 1 missing bit and refuses the irrelevant one. On the adder, without being told, it finds the carry as the 1-bit ledger, and it adds nothing where there is no future.
- `future_classes` reproduces the corpus cut law: the gap threshold $\lceil\log_b m\rceil$ and the failure counts.
- `transport` classifies receiver capacity against the written ledger as exclusion, faithful, or invention, with $\ell$ computed rather than supplied.

The partition refinement is Moore's algorithm; the minimality of the coarsest closed refinement is the Myhill–Nerode theorem. The contribution here is the runtime contract, not the algorithm.